# HemoDyn-AI v2: Physics-Guided Machine Learning (PGML) Surrogate Modeling
**Author**: Jace E. Roman (Temple University)  
**Contact**: jace.roman@temple.edu  
**License**: MIT

---

### Overview
Standard black-box machine learning approaches suffer severe errors in high-gradient hemodynamic regimes (such as arterial stenosis throats) where unconstrained models over-predict velocity peaks or violate mass conservation.

**HemoDyn-AI** introduces a **Physics-Guided Residual Surrogate**:
$$\Delta u(\mathbf{x}) = u_{\text{CFD}}(\mathbf{x}) - u_{\text{physics}}(\mathbf{x})$$
$$u_{\text{pred}}(\mathbf{x}) = \max\left(0, u_{\text{physics}}(\mathbf{x}) + \Delta \hat{u}(\mathbf{x})\right)$$

The analytical prior $u_{\text{physics}}(\mathbf{x})$ enforces continuity-driven laminar Poiseuille flow, and the machine learning model focuses strictly on predicting the viscous/separation discrepancy.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path("..").resolve() / "src"))
from hemodyn.data import load_hemodyn_dataset, get_spatial_holdout_split
from hemodyn.surrogate import PhysicsGuidedSurrogate, GeometryOnlySurrogate, evaluate_surrogate_performance
from hemodyn.plotting import set_publication_style

set_publication_style()
print("Loaded hemodyn package.")

### 1. Loading the 132,740-Point CFD Benchmark Dataset

In [ ]:
data = load_hemodyn_dataset("../results/data/hemodyn_training_data.npz")
X = data["X"]
y = data["y"]
u_phys = data["u_phys"]
residual = data["residual"]

print(f"Total fluid mesh nodes : {len(y):,}")
print(f"Feature matrix shape   : {X.shape}")
print(f"Target velocity range  : [{y.min():.2f}, {y.max():.2f}] mm/s")
print(f"Physics prior range    : [{u_phys.min():.2f}, {u_phys.max():.2f}] mm/s")
print(f"Residual range         : [{residual.min():.2f}, {residual.max():.2f}] mm/s")

### 2. Spatial Holdout Partitioning ($z \in [55, 75]\text{ mm}$)
We partition the dataset so that the high-gradient stenosis narrowing is held out from training.

In [ ]:
X_train, y_train, u_train, X_test, y_test, u_test = get_spatial_holdout_split(data, 55.0, 75.0)

print(f"Training set points : {len(y_train):,}")
print(f"Test set points     : {len(y_test):,}")

### 3. Training Surrogates & Verifying Throat Error Reduction
We train both the standard **Geometry-Only ML** surrogate and the **Physics-Guided Residual ML** surrogate.

In [ ]:
# Train Geometry-Only Surrogate
geom_model = GeometryOnlySurrogate(max_iter=100, random_state=42)
geom_model.fit(X_train, y_train)
geom_metrics = geom_model.evaluate(X_test, y_test)

# Train Physics-Guided Residual Surrogate
pgml_model = PhysicsGuidedSurrogate(max_iter=100, random_state=42)
pgml_model.fit(X_train, y_train, u_train)
pgml_metrics = pgml_model.evaluate(X_test, y_test, u_test)

# Physics baseline metrics
phys_metrics = evaluate_surrogate_performance(y_test, u_test)

print("=== SPATIAL HOLDOUT BENCHMARK RESULTS ===")
print(f"Physics Prior    : RMSE = {phys_metrics['rmse']:.3f} mm/s | MAE = {phys_metrics['mae']:.3f} mm/s | R2 = {phys_metrics['r2']:.4f}")
print(f"Geometry-Only ML : RMSE = {geom_metrics['rmse']:.3f} mm/s | MAE = {geom_metrics['mae']:.3f} mm/s | R2 = {geom_metrics['r2']:.4f}")
print(f"Physics-Guided ML: RMSE = {pgml_metrics['rmse']:.3f} mm/s | MAE = {pgml_metrics['mae']:.3f} mm/s | R2 = {pgml_metrics['r2']:.4f}")

### 4. Parity Plot: CFD Ground Truth vs. Surrogate Predictions

In [ ]:
y_pred_geom = geom_model.predict(X_test)
y_pred_pgml = pgml_model.predict(X_test, u_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Subsample for clean scatter
idx = np.random.RandomState(42).choice(len(y_test), size=min(3000, len(y_test)), replace=False)

axes[0].scatter(y_test[idx], y_pred_geom[idx], alpha=0.3, s=10, color="#d95f02")
axes[0].plot([0, 1000], [0, 1000], 'k--', alpha=0.7)
axes[0].set_xlabel("CFD Ground Truth (mm/s)")
axes[0].set_ylabel("Predicted Velocity (mm/s)")
axes[0].set_title(f"Geometry-Only ML\nRMSE = {geom_metrics['rmse']:.2f} mm/s")

axes[1].scatter(y_test[idx], y_pred_pgml[idx], alpha=0.3, s=10, color="#1b7837")
axes[1].plot([0, 1000], [0, 1000], 'k--', alpha=0.7)
axes[1].set_xlabel("CFD Ground Truth (mm/s)")
axes[1].set_title(f"Physics-Guided ML (Ours)\nRMSE = {pgml_metrics['rmse']:.2f} mm/s")

plt.tight_layout()
plt.show()